# Laboratorium 3: wybór modelu walidacją krzyżową i wyciek informacji

Karta laboratorium: <https://rodakt.github.io/MAD/laby/03-walidacja-krzyzowa.html>

Treść zadań jest w książce, tutaj jest tylko szkielet kodu. Miejsca do uzupełnienia oznaczono `...`.
Notebook otwarty z linku na karcie laboratorium nie jest zapisany na Twoim Dysku: po ponownym kliknięciu linku zobaczysz wersję wyjściową.
Na początku zajęć zapisz kopię (*Plik → Zapisz kopię na Dysku*) i pracuj na niej.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2026)

def new_seed():
    """Losowe ziarno dla funkcji scikit-learn, które przyjmują random_state."""
    return int(rng.integers(2**32))

## Część 1. Wybór $k$ walidacją krzyżową (Zadanie 5.8)

Dane to dwa półksiężyce z szumem $0{,}3$ (`make_moons`): dwie cechy, dwie klasy. Próba ucząca ma $n=200$ obserwacji.
Ryzyko, jak w poprzednich laboratoriach, przybliżamy średnią stratą na dużej nowej próbie ($20\,000$ obserwacji).
Kandydaci to klasyfikatory $k$-NN dla $k=1,3,\dots,99$; nieparzyste $k$ wykluczają remisy przy dwóch klasach.
Na końcu tej listy są modele najprostsze: im większe $k$, tym silniejsze wygładzanie.

In [ ]:
def generate_data(n):
    return make_moons(n_samples=n, noise=0.3, random_state=new_seed())

X_tr, Y_tr = generate_data(200)
X_te, Y_te = generate_data(20_000)
k_values = np.arange(1, 100, 2)

plt.scatter(X_tr[:, 0], X_tr[:, 1], c=Y_tr, cmap="coolwarm", s=15)
plt.title("próba ucząca, $n = 200$")
plt.show()

### 1.1. Predykcje dla wszystkich $k$ naraz

Będziemy liczyć błędy 50 kandydatów, w części 1.3 dla 100 prób. Dopasowanie każdego klasyfikatora osobno trwałoby
kilka minut, więc korzystamy z tego, że dla ustalonego punktu $\mathbf{x}$ jego $k$ najbliższych sąsiadów w próbie
uczącej to po prostu pierwsze $k$ pozycji listy posortowanej od najbliższego -- ta sama lista obsługuje każde $k$.
`NearestNeighbors` znajduje więc raz 99 najbliższych sąsiadów każdego punktu, w którym robimy predykcję, a `np.cumsum` po wierszach liczy,
ilu spośród $j$ najbliższych ma klasę 1, dla wszystkich $j$ naraz. Klasyfikator $k$-NN wskazuje klasę 1, gdy ta liczba
dla $j=k$ przekracza $k/2$. Funkcja jest gotowa; sprawdź na jednym $k$, że daje to samo co `KNeighborsClassifier`.

In [ ]:
def knn_predict_all(X_tr, Y_tr, X_new, k_values):
    """Predykcje k-NN w punktach X_new dla wszystkich k z k_values; wiersz j -- dla k_values[j]."""
    k_max = k_values.max()
    nn = NearestNeighbors(n_neighbors=k_max).fit(X_tr)   # struktura do szukania sąsiadów w próbie uczącej
    idx = nn.kneighbors(X_new, return_distance=False)    # idx[i]: indeksy k_max sąsiadów punktu X_new[i], od najbliższego
    neighbor_classes = Y_tr[idx]                         # klasy tych sąsiadów, ten sam kształt co idx
    ones = np.cumsum(neighbor_classes, axis=1)           # ones[i, j-1]: liczba jedynek wśród j najbliższych sąsiadów punktu i
    ones_k = ones[:, k_values - 1]                       # kolumny dla j = k, po jednej na każde k z k_values
    pred = (ones_k > k_values / 2).astype(int)           # głosowanie większościowe; pred[i, j] -- dla k = k_values[j]
    return pred.T

pred_all = knn_predict_all(X_tr, Y_tr, X_te, k_values)
j = 7   # k = 15
pred_sklearn = KNeighborsClassifier(n_neighbors=k_values[j]).fit(X_tr, Y_tr).predict(X_te)
print(pred_all.shape, np.mean(pred_all[j] != pred_sklearn))

### 1.2. Walidacja krzyżowa a ryzyko na jednej próbie

`StratifiedKFold(n_splits=10, shuffle=True, random_state=...)` losuje podział na 10 części warstwowo, czyli z tymi samymi
proporcjami klas w każdej części. Jego metoda `split(X, Y)` zwraca kolejno pary `(tr, te)`: indeksy obserwacji, na których
uczymy, i indeksy części oceniającej.
Funkcja `cv_errors` zapisuje stratę 0--1 każdej obserwacji w chwili, gdy jest w części oceniającej, osobno dla każdego $k$.
Średnia z wiersza to $\mathrm{CV}_{10}$, a błąd standardowy liczymy naiwnie, jak w książce
([Błąd standardowy walidacji krzyżowej](https://rodakt.github.io/MAD_book/1-fundamenty/05-ocena-modeli.html#błąd-standardowy-walidacji-krzyżowej)):
odchylenie standardowe $n$ strat podzielone przez $\sqrt n$.

Reguła jednego błędu standardowego wybiera najprostszego kandydata (największe $k$), którego $\mathrm{CV}_{10}$ nie przekracza
minimum o więcej niż jeden błąd standardowy. Przyda się `np.nonzero(warunek)[0]` -- indeksy, w których warunek jest spełniony.

In [ ]:
def cv_errors(X, Y, k_values, cv):
    """Oszacowania walidacji krzyżowej z podziałem cv i ich naiwne błędy standardowe dla wszystkich k."""
    losses = np.empty((len(k_values), len(Y)))   # wiersz j: straty 0-1 dla k_values[j]
    for tr, te in cv.split(X, Y):
        losses[:, te] = ...   # uczymy na obserwacjach tr, oceniamy na te
    return ..., ...           # średnia i błąd standardowy w każdym wierszu

def risks(X_tr, Y_tr, k_values):
    """Ryzyko wszystkich kandydatów, przybliżone średnią stratą na nowej próbie (X_te, Y_te)."""
    return ...

def choose_k(cv_err, cv_se):
    """Indeksy kandydatów wybranych przez minimum CV i regułą jednego błędu standardowego."""
    i_min = ...
    i_1se = ...
    return i_min, i_1se

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=new_seed())
cv_err, cv_se = cv_errors(X_tr, Y_tr, k_values, cv)
risk = risks(X_tr, Y_tr, k_values)
i_min, i_1se = choose_k(cv_err, cv_se)
i_best = np.argmin(risk)

plt.plot(k_values, cv_err, "o-", ms=3, label="$\\mathrm{CV}_{10}$")
plt.fill_between(k_values, cv_err - cv_se, cv_err + cv_se, alpha=0.2)
plt.plot(k_values, risk, "k-", label="ryzyko (nowa próba)")
for i, color, name in [(i_min, "C1", "minimum CV"), (i_1se, "C2", "reguła 1 SE"), (i_best, "k", "najmniejsze ryzyko")]:
    plt.axvline(k_values[i], color=color, ls="--", lw=1, label=f"{name}: $k = {k_values[i]}$")
plt.xlabel("$k$")
plt.ylabel("błąd klasyfikacji")
plt.legend()
plt.show()

Sprawdź `cv_errors` biblioteką scikit-learn: `cross_val_predict(model, X, Y, cv=cv)` zwraca dla każdej obserwacji
predykcję modelu, który jej nie widział. Obiekt `cv` ze stałym `random_state` daje przy każdym wywołaniu ten sam podział,
więc dla $k=15$ średnia strata z tych predykcji powinna być równa elementowi `cv_err` dla $k=15$.

In [ ]:
...

Pytania:

1. Jak daleko od siebie są $k$ wybrane minimum CV, regułą jednego błędu standardowego i $k$ o najmniejszym ryzyku?
   Jak duża jest różnica ich ryzyk w porównaniu z błędem standardowym?
2. Jak wygląda krzywa $\mathrm{CV}_{10}$ w otoczeniu minimum? Dlaczego przy płaskiej krzywej wybór konkretnego $k$
   jest niestabilny, a wybór dobrego modelu -- nie?
3. Uruchom komórkę jeszcze raz (nowa próba). Co się zmienia?

### 1.3. Sto prób (punkty b i c)

Dla 100 prób uczących zapisz: $k$ wybrane obiema regułami, stratę wynikającą z wyboru, czyli ryzyko wybranego modelu
minus najmniejsze ryzyko wśród kandydatów, oraz różnicę między najmniejszym wynikiem walidacji krzyżowej a ryzykiem
wybranego modelu. Pętla trwa kilkadziesiąt sekund.

In [ ]:
n_reps = 100
k_min, k_1se = np.empty(n_reps, dtype=int), np.empty(n_reps, dtype=int)
loss_min, loss_1se = np.empty(n_reps), np.empty(n_reps)
optimism = np.empty(n_reps)   # min CV minus ryzyko modelu wybranego przez minimum CV
for r in range(n_reps):
    X_tr, Y_tr = generate_data(200)
    ...

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
bins = np.arange(0, 102, 4)
for ax, ks, name in [(axes[0], k_min, "minimum CV"), (axes[1], k_1se, "reguła 1 SE")]:
    ax.hist(ks, bins=bins)
    ax.set_title(name)
    ax.set_xlabel("wybrane $k$")
axes[0].set_ylabel("liczba prób")
plt.show()

print(f"średnia strata z wyboru: minimum CV {loss_min.mean():.4f}, reguła 1 SE {loss_1se.mean():.4f}")
print(f"średnio min CV minus ryzyko wybranego modelu: {optimism.mean():.4f} "
      f"(błąd std. {optimism.std() / np.sqrt(n_reps):.4f})")

Pytania:

1. Która reguła wybiera $k$ stabilniej? Która ma mniejszą średnią stratę z wyboru? Porównaj tę stratę z najmniejszym ryzykiem
   (około $0{,}09$) i z błędem standardowym oszacowania z 1.2.
2. Jaki znak ma średnia różnica między minimum CV a ryzykiem wybranego modelu? Odnieś wynik do
   [twierdzenia o optymizmie wyboru](https://rodakt.github.io/MAD_book/1-fundamenty/05-ocena-modeli.html#thm-optymizm-wyboru).
3. Walidacja krzyżowa uczy modele na $9/10$ próby, więc dla jednego kandydata jest raczej pesymistyczna. Która z dwóch tendencji
   wygrywa tutaj?

## Część 2. Wyciek informacji (Zadanie 5.7)

Powtarzamy przykład z [sekcji o wycieku](https://rodakt.github.io/MAD_book/1-fundamenty/05-ocena-modeli.html#sec-wyciek):
$n=50$ obserwacji z dwóch równolicznych klas i $p=5000$ cech, które są czystym szumem. Każdy klasyfikator ma tu ryzyko $1/2$.
Procedura: wybór $q=100$ cech najsilniej związanych z klasą, potem klasyfikator 1-NN.

W książce cechy wybieraliśmy według korelacji z klasą. `SelectKBest(f_classif, k=q)` wybiera $q$ cech o największej
statystyce $F$ jednoczynnikowej analizy wariancji; przy dwóch klasach jest ona rosnącą funkcją modułu korelacji,
więc wybór jest ten sam. Jak każdy krok scikit-learn ma metody `fit` i `transform` (albo obie naraz: `fit_transform`).

### 2.1. Zły i dobry sposób (punkt b)

W złym sposobie cechy wybieramy raz, na całej próbie, i walidacji poddajemy tylko klasyfikator. W dobrym wybór cech
jest pierwszym krokiem potoku `make_pipeline(SelectKBest(...), KNeighborsClassifier(...))`, a `cross_val_score`
wywołuje `fit` całego potoku osobno w każdej części. `cross_val_score` zwraca trafność (odsetek poprawnych klasyfikacji)
w każdej części; błąd to jeden minus jej średnia. W obu sposobach użyj tego samego podziału `cv`.

In [ ]:
def noise_data(n, p):
    return rng.normal(size=(n, p)), np.repeat([0, 1], n // 2)

n_reps = 50
bad, good = np.empty(n_reps), np.empty(n_reps)
for r in range(n_reps):
    X, Y = noise_data(50, 5000)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=new_seed())
    X_sel = ...   # 100 cech wybranych na całej próbie
    bad[r] = ...
    pipe = ...
    good[r] = ...

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
bins = np.linspace(0, 0.9, 46)
for ax, est, name in [(axes[0], bad, "wybór cech przed walidacją"), (axes[1], good, "wybór cech w potoku")]:
    ax.hist(est, bins=bins)
    ax.axvline(0.5, color="red", ls="--")
    ax.set_title(name)
    ax.set_xlabel("oszacowanie błędu walidacją krzyżową")
axes[0].set_ylabel("liczba prób")
plt.show()
print(f"średnie oszacowanie: zły sposób {bad.mean():.3f}, potok {good.mean():.3f}")

### 2.2. Od czego zależy wyciek (punkt a)

Dla złego sposobu oblicz średnie oszacowanie (z 20 prób) dla $q\in\{1,10,100,1000\}$ i $p\in\{100,1000,5000\}$
i wypisz je w tabeli: wiersz to $p$, kolumna to $q$. Pomiń kombinacje z $q>p$. Kod z 2.1 wystarczy przenieść
do funkcji z argumentami `p`, `q`.

In [ ]:
...

Pytania:

1. Dlaczego wyciek rośnie z $p$? Ile wynosi oszacowanie dla $q=p=100$ i dlaczego?
2. Dlaczego zależność od $q$ nie jest monotoniczna? Co się dzieje przy $q=1$, a co przy $q$ bliskim $p$?

### 2.3. Standaryzacja (punkt c)

Zamiast wyboru cech zastosuj standaryzację (`StandardScaler`) przed klasyfikatorem 1-NN: raz wyznaczoną na całej próbie,
raz w potoku. Dla tych samych prób i podziałów oblicz różnicę oszacowań i jej średnią z 50 prób z błędem standardowym.
Czy ten wyciek ma mierzalny wpływ? Porównaj z 2.1 i z [akapitem w książce](https://rodakt.github.io/MAD_book/1-fundamenty/05-ocena-modeli.html#sec-wyciek),
który wyjaśnia, dlaczego przy standaryzacji wyciek jest zwykle niewielki.

In [ ]:
...